<a href="https://colab.research.google.com/github/4cekay/B101-Group2-NLP-Project/blob/backup-points/Troubleshooting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy torch datasets transformers sentencepiece protobuf accelerate evaluate tensorboard scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [1]:
import os
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, IntervalStrategy, EarlyStoppingCallback
import torch
from sklearn.model_selection import train_test_split
import huggingface_hub
from collections import Counter
from sklearn.metrics import roc_curve, auc, roc_auc_score, f1_score, confusion_matrix, precision_recall_curve, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import json


In [3]:
# 1. Load the final saved model + tokenizer
checkpoint_dir = "./runs/deberta_v3_interpolate_test_v2/checkpoint-650"   # or your groupmate’s saved folder
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir, dtype=torch.float32)
model.eval()

# 2. Load your evaluation set
external_df = pd.read_csv("./dataset/formal_val_1350.csv", encoding="utf-8")
external_dataset = Dataset.from_pandas(external_df)

# 3. Tokenize with the same settings used in training
MAX_LENGTH = 350
def tokenize(example):
    tokenized = tokenizer(
        example["sample_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    tokenized["labels"] = example["AI_label"]
    return tokenized

tokenized_eval_dataset = external_dataset.map(
    tokenize,
    batched=True,
    remove_columns=external_dataset.column_names
)

# 4. Create a minimal Trainer for evaluation
eval_args = TrainingArguments(
    output_dir="./tmp_eval",
    per_device_eval_batch_size=16,
    report_to="none"
)

eval_trainer = Trainer(
    model=model,
    args=eval_args,
)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

In [4]:
# 5. Run prediction
preds_output = eval_trainer.predict(tokenized_eval_dataset)
logits = preds_output.predictions
labels = preds_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
prob_ai = probs[:, 1]  # probability of AI-generated
predicted_classes = np.argmax(logits, axis=-1)

# 6. Print metrics
print("True label distribution:", Counter(labels.tolist()))
print("Predicted label distribution:", Counter(predicted_classes.tolist()))
print("AUC:", roc_auc_score(labels, prob_ai))
print("F1:", f1_score(labels, predicted_classes))
print("Confusion matrix:\n", confusion_matrix(labels, predicted_classes))


True label distribution: Counter({0: 675, 1: 675})
Predicted label distribution: Counter({0: 1350})
AUC: 0.10258655692729766
F1: 0.0
Confusion matrix:
 [[675   0]
 [675   0]]


In [ ]:
checkpoint_dir = r"C:\temp\decisive_check"
tokenizer2 = AutoTokenizer.from_pretrained(checkpoint_dir)
model2 = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir, dtype=torch.float32)
model2.eval()

# sanity check: do the two tokenizers actually agree?
test_text = external_df["sample_text"].iloc[0]
print("original notebook tokenizer:", tokenizer(test_text)["input_ids"][:15])
print("reloaded tokenizer:        ", tokenizer2(test_text)["input_ids"][:15])
# any mismatch here is your answer on the tokenizer question, immediately